# 10.7 Standard and built-in table handlers

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To work with external database files, AMPL relies on `table` handlers. These are addons,
usually in the form of shared or dynamic link libraries, that can be loaded as needed.
**Figure 10-8:** Another two-dimensional Excel `table`.

AMPL is distributed with a "standard" `table` handler that runs under Microsoft Windows
and communicates via the Open Database Connectivity (ODBC) application programming
interface; it recognizes relational tables in the formats used by Access, Excel, and any
other application for which an ODBC driver exists on your computer. Additional handlers
may be supplied by vendors of AMPL or of database software.

In addition to any supplied handlers, minimal ASCII and `binary` relational `table` file
handlers are built into AMPL for testing. Vendors may include other built-in handlers. If
you are not sure which handlers are currently seen by your copy of AMPL, the features
described in A.13 can get you a list of active handlers and brief instructions for using
them.

As the introductory examples of this chapter have shown, AMPL communicates with
handlers through the string-list in the `table` declaration. The form and interpretation of
the string-list are specific to each handler. The remainder of this section describes the
string-lists that are recognized by AMPL's standard ODBC handler. Following a general
introduction, specific instructions are provided for the two applications, Access and
Excel, that are used in many of the examples in preceding sections. A final subsection
describes the string-lists recognized by the built-in `binary` and ASCII `table` handlers.

Using the standard ODBC `table` handler

In the context of a declaration that begins `table` `table`-name ..., the general form of
the string-list for the standard ODBC `table` handler is

```ampl
"ODBC" "connection-spec" "external-table-spec" opt  "verbose" opt
```

The first string tells AMPL that `data` transfers using this `table` declaration should employ
the standard ODBC handler. Subsequent strings then provide directions to that handler.

The second string identifies the external database file that is to be read or written upon
execution of `read table` `table`-name or `write table` `table`-name commands. There
are several possibilities, depending on the form of the connection-spec and the configuration
of ODBC on your computer.

If the connection-spec is a filename of the form name.ext, where ext is a 3-letter
extension associated with an installed ODBC driver, then the named file is the database
file. This form can be seen in a number of our examples, where filenames of the forms
name.mdb and name.xls refer to Access and Excel files, respectively.

Other forms of connection-spec are more specific to ODBC, and are explained in
online documentation. Information about your computer's configuration of ODBC
drivers, `data` source names, file `data` sources, and related entities can be examined and
changed through the Windows ODBC control panel.

The third string normally gives the name of the relational `table`, within the specified
file, that is to be read or written upon execution of `read table` or `write table` commands
. If the third string is omitted, the name of the relational `table` is taken to be the
same as the `table`-name of the containing `table` declaration. For writing, if the indicated
`table` does not exist, it is created; if the `table` exists but all of the `table`
declaration's `data`-specs have read/write status OUT, then it is overwritten. Otherwise,
writing causes the existing `table` to be modified; each column written either overwrites an
existing column of the same name, or becomes a new column appended to the `table`.

Alternatively, if the third string has the special form

```ampl
"SQL=sql-query"
```

the `table` declaration applies to the relational `table` that is (temporarily) created by a statement
in the Structured Query Language, commonly abbreviated SQL. Specifically, a relational
`table` is first constructed by executing the SQL statement given by sql-query, with
respect to the database file given by the second string in the `table` declaration's stringlist.
Then the usual interpretations of the `table` declaration are applied to the constructed
`table`. All columns specified in the declaration should have read/write status IN,
since it would make no sense to write to a temporary `table`. Normally the sql-query is a
SELECT statement, which is SQL's primary device for operating on tables to create new
ones.

As an example, if you wanted to read as `data` for diet.mod only those foods having
a cost of $2.49 or less, you could use an SQL query to extract the relevant records from
the Foods `table` of your database:

```ampl
table cheapFoods IN "ODBC" "diet.mdb"
   "SQL=SELECT * FROM Foods WHERE cost <= 2.49":
   FOOD <- [FOOD], cost, f_min, f_max;
```

Then to read the relevant `data` for parameter amt, which is indexed over nutrients and
foods, you would want to read only those records that pertained to a food having a cost of
$2.49 or less. Here is one way that an SQL query could be used to extract the desired
records:

```ampl
option selectAmts "SELECT NUTR, Amts.FOOD, amt "
   "FROM Amts, Foods "
      "WHERE Amts.FOOD = Foods.FOOD and cost <= 2.49";
table cheapAmts IN "ODBC" "diet.mdb" ("SQL=" & $selectAmts):
   [NUTR, FOOD], amt;
```

Here we have used an AMPL option to store the string containing the SQL query. Then
the `table` declaration's third string can be given by the relatively short string expression
"SQL=" & $selectAmts.

The string verbose after the first three strings requests diagnostic messages — such
as the DSN= string that ODBC reports using — whenever the containing `table` declaration
is used by a `read table` or `write table` command.

Using the standard ODBC `table` handler with Access and Excel
To set up a relational `table` correspondence for reading or writing Microsoft Access
files, specify the ext in the second string of the string-list as mdb:

```ampl
"ODBC" "filename.mdb" "external-table-spec" opt
```

The file named by the second string must exist, though for writing it may be a database
that does not yet contain any tables.

To set up a relational `table` correspondence for reading or writing Microsoft Excel
spreadsheets, specify the ext in the second string of the string-list as xls:

```ampl
"ODBC" "filename.xls" "external-table-spec" opt
```

In this case, the second string identifies the external Excel workbook file that is to be read
or written. For writing, the file specified by the second string is created if it does not
exist already.

The external-`table`-spec specified by the third string identifies a spreadsheet range,
within the specified file, that is to be read or written; if this string is absent, it is taken to
be the `table`-name given at the start of the `table` declaration. For reading, the specified
range must exist in the Excel file. For writing, if the range does not exist, it is created, at
the upper left of a new worksheet having the same name. If the range exists but all of the
`table` declaration's `data`-specs have read/write status OUT, it is overwritten. Otherwise,
writing causes the existing range to be modified. Each column written either overwrites
an existing column of the same name, or becomes a new column appended to the `table`;
each row written either overwrites entries in an existing row having the same key column
entries, or becomes a new row appended to the `table`.

When writing causes an existing range to be extended, rows or columns are added at
the bottom or right of the range, respectively. The cells of added rows or columns must
be empty; otherwise, the attempt to write the `table` fails and the `write table` command
elicits an error message. After a `table` is successfully written, the corresponding range is
created or adjusted to contain exactly the cells of that `table`.
Built-in `table` handlers for text and `binary` files
For debugging and demonstration purposes, AMPL has built-in handlers for two very
simple relational `table` formats. These formats store one `table` per file and convey equivalent
information. One produces ASCII files that can be examined in any text editor, while
the other creates `binary` files that are much faster to read and write.

For these handlers, the `table` declaration's string-list contains at most one string,
identifying the external file that contains the relational `table`. If the string has the form

```ampl
"filename.tab"
```

the file is taken to be an ASCII text file; if it has the form

```ampl
"filename.bit"
```

it is taken to be a `binary` file. If no string-list is given, a text file `table`-name.tab is
assumed.

For reading, the indicated file must exist. For writing, if the file does not exist, it is
created. If the file exists but all of the `table` declaration's `data`-specs have read/write
status OUT, it is overwritten. Otherwise, writing causes the existing file to be modified;
each column written either replaces an existing column of the same name, or becomes a
new column added to the `table`.

The format for the text files can be examined by writing one and viewing the results
in a text editor. For example, the following AMPL session,

```ampl
ampl: model diet.mod;
ampl: data diet2a.dat;
ampl: solve;
MINOS 5.5: optimal solution found.
13 iterations, objective 118.0594032
ampl: table ResultList OUT "DietOpt.tab":
ampl?    [FOOD], Buy, Buy.rc, {j in FOOD} Buy[j]/f_max[j];
ampl: write table ResultList;
```

produces a file DietOpt.tab with the following content:

```ampl
ampl.tab 1 3
FOOD Buy Buy.rc 'Buy[j]/f_max[j]'
BEEF 5.360613810741701 8.881784197001252e-16 0.5360613810741701
CHK 2 1.1888405797101402 0.2
FISH 2 1.1444075021312856 0.2
HAM 10 -0.30265132139812223 1
MCH 10 -0.5511508951406658 1
MTL 10 -1.3289002557544745 1
SPG 9.306052855924973 -8.881784197001252e-16 0.9306052855924973
TUR 1.9999999999999998 2.7316197783461176 0.19999999999999998
```

In the first line, ampl.tab identifies this as an AMPL relational `table` text file, and is followed
by the numbers of key and non-key columns, respectively. The second line gives
the names of the table's columns, which may be any strings. (Use of the ˜ operator to
specify valid column-names is not necessary in this case.) Each subsequent line gives the
values in one `table` row; numbers are written in full precision, with no special formatting
or alignment.